# 10 — Sequential DoE: augment and propose next runs

Close the loop: start from a weak design, propose the next batch, and ask whether the extra runs are worth it (Δ D/G/SPV/power).

In [ ]:
import numpy as np
import doekit as ed
print("doekit", ed.__version__)

## 1. Weak starting design

In [ ]:
facs = [ed.ContinuousFactor("x1", -1, 1), ed.ContinuousFactor("x2", -1, 1)]
base = ed.random_design(facs, n=6, seed=0)
base.model = ed.Model.parse("0 ~ x1 + x2 + x1:x2")
print(ed.evaluate(base, n_region=2000, seed=0).summary())

## 2. Propose next runs (no response yet)

In [ ]:
prop = ed.propose_next_runs(base, n_add=4, criterion="D", n_candidates=120, seed=1)
print(prop.rationale)
print(prop.comparison.summary)
display(prop.comparison.table)
display(prop.added.matrix)

## 3. With simulated responses

In [ ]:
rng = np.random.default_rng(2)
X = base.model.matrix(base.matrix)
beta = np.array([1.0, 2.0, -1.5, 0.8])
y = X @ beta + rng.normal(0, 0.3, base.n_runs)
prop2 = ed.propose_next_runs(base, response=y, n_add=4, budget=16, seed=3)
print("sigma_hat", prop2.sigma_hat)
print("active", prop2.active_terms)
print(prop2.comparison.summary)
print(prop2.to_dict()["schema"])

## 4. BO bridge (bounds → candidates)

In [ ]:
cand = ed.candidates_from_bounds([("x1", -1, 1), ("x2", -1, 1)], n=100, seed=4)
aug = ed.augment_design(base, n_add=3, candidates=cand, criterion="I", seed=5)
print(ed.compare_designs(base, aug, n_region=1000, seed=0).summary)